# Decision Tree from scratch implementation

### Imports

Loads the core libraries needed for the whole notebook — **pandas** for tabular data handling, **numpy** for the vectorized math behind the model, **Counter** for majority voting, **train_test_split** for training, and **load_breast_cancer** from scikit-learn as a real-data source.

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

### Loading Data

Loads the real Breast Cancer Wisconsin dataset using all 30 real tumor measurements as features, converts numeric diagnosis codes into readable labels, and splits into training and test sets.

In [2]:
data = load_breast_cancer(as_frame=True)
df = data.frame

X = df.drop(columns=["target"]).values.astype(float)
y = np.array(["malignant" if t == 0 else "benign" for t in df["target"].values])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
feature_names = list(df.drop(columns=["target"]).columns)

print(f"Training examples: {X_train.shape[0]}, Test examples: {X_test.shape[0]}, Features: {X_train.shape[1]}")

Training examples: 455, Test examples: 114, Features: 30


### Node Class

Defines the building block of the tree. Each Node is either a decision node (holding a feature index, a threshold to split on, and pointers to a left/right child) or a leaf node (holding a final predicted class and no children).


In [3]:
class Node:
    def __init__(self, feature_idx=None, threshold=None, info_gain=None, left=None, right=None, value=None):
        # Decision node
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.info_gain = info_gain
        self.left = left
        self.right = right

        # Leaf node
        self.value = value

### Decision Tree Class

Implements the tree-building algorithm from scratch: recursively finds the feature and threshold that best separates the two classes (by maximizing information gain / entropy reduction), splits the data accordingly, and repeats on each half until hitting **max_depth** or running out of samples to split. **predict_class** walks a new patient down the tree until it lands on a leaf's predicted class.

In [4]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=3):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def build_tree(self, X, y, curr_depth=0):
        n_samples, n_features = X.shape

        if n_samples >= self.min_samples_split and curr_depth <= self.max_depth:
            best_split = self.best_split(X, y, n_features)

            if best_split["info_gain"] > 0:
                left_node = self.build_tree(best_split["X_left"], best_split["y_left"], curr_depth + 1)
                right_node = self.build_tree(best_split["X_right"], best_split["y_right"], curr_depth + 1)
                return Node(best_split["feature_idx"], best_split["threshold"],
                            best_split["info_gain"], left_node, right_node)

        leaf_value = Counter(y).most_common(1)[0][0]
        return Node(value=leaf_value)

    def best_split(self, X, y, n_features):
        best = {'feature_idx': None, 'threshold': None, 'info_gain': -1,
                'X_left': None, 'y_left': None, 'X_right': None, 'y_right': None}

        for feature_idx in range(n_features):
            thresholds = np.unique(X[:, feature_idx])

            for threshold in thresholds:
                mask = X[:, feature_idx] <= threshold
                X_left, y_left = X[mask], y[mask]
                X_right, y_right = X[~mask], y[~mask]

                if len(y_left) and len(y_right):
                    info_gain = self.information_gain(y, y_left, y_right)

                    if info_gain > best['info_gain']:
                        best.update(feature_idx=feature_idx, threshold=threshold, info_gain=info_gain,
                                    X_left=X_left, y_left=y_left, X_right=X_right, y_right=y_right)

        return best

    def information_gain(self, parent_y, left_y, right_y):
        left_weight = len(left_y) / len(parent_y)
        right_weight = len(right_y) / len(parent_y)
        return self.entropy(parent_y) - (left_weight * self.entropy(left_y) + right_weight * self.entropy(right_y))

    def entropy(self, y):
        entropy = 0
        for class_label in np.unique(y):
            p = len(y[y == class_label]) / len(y)
            entropy += -p * np.log2(p)
        return entropy

    def fit(self, X, y):
        self.root = self.build_tree(X, y)

    def predict(self, X):
        return np.array([self.predict_class(row, self.root) for row in X])

    def predict_class(self, row, node):
        if node.value is not None:
            return node.value
        if row[node.feature_idx] <= node.threshold:
            return self.predict_class(row, node.left)
        else:
            return self.predict_class(row, node.right)

### Training & Predicting

Builds the tree on the training data (**max_depth=3**), then predicts on the 114 held-out test patients and measures accuracy.

In [5]:
dt = DecisionTree(min_samples_split=2, max_depth=3)
dt.fit(X_train, y_train)

predictions = dt.predict(X_test)
accuracy = np.mean(predictions == y_test) * 100
print(f"Test accuracy: {accuracy:.2f}%")

Test accuracy: 91.23%


**91.23% test accuracy** — 104 of 114 real, previously unseen patients correctly classified.

### Visualizing the Tree

Attaches a **print_tree** method to the **DecisionTree** class and prints the actual learned tree structure, using real feature names instead of generic indices, so each split is interpretable.


In [10]:
def print_tree(self, node=None, depth=0, indent="|   ", feature_names=None):
    prefix = indent * depth
    if node is None:
        node = self.root

    if node.value is not None:
        print(f"{prefix}|--- class: {node.value}")
        return

    label = feature_names[node.feature_idx] if feature_names is not None else f"Feature {node.feature_idx}"
    print(f"{prefix}|--- {label} <= {node.threshold:.4f}")
    self.print_tree(node.left, depth + 1, indent, feature_names)
    print(f"{prefix}|--- {label} > {node.threshold:.4f}")
    self.print_tree(node.right, depth + 1, indent, feature_names)

DecisionTree.print_tree = print_tree

dt.print_tree(feature_names=feature_names)

|--- worst perimeter <= 105.9000
|   |--- worst concave points <= 0.1342
|   |   |--- area error <= 43.5200
|   |   |   |--- worst texture <= 33.1700
|   |   |   |   |--- class: benign
|   |   |   |--- worst texture > 33.1700
|   |   |   |   |--- class: benign
|   |   |--- area error > 43.5200
|   |   |   |--- mean radius <= 11.7600
|   |   |   |   |--- class: malignant
|   |   |   |--- mean radius > 11.7600
|   |   |   |   |--- class: benign
|   |--- worst concave points > 0.1342
|   |   |--- worst texture <= 27.2000
|   |   |   |--- worst area <= 719.8000
|   |   |   |   |--- class: benign
|   |   |   |--- worst area > 719.8000
|   |   |   |   |--- class: malignant
|   |   |--- worst texture > 27.2000
|   |   |   |--- class: malignant
|--- worst perimeter > 105.9000
|   |--- worst perimeter <= 115.9000
|   |   |--- worst texture <= 19.8500
|   |   |   |--- class: benign
|   |   |--- worst texture > 19.8500
|   |   |   |--- mean smoothness <= 0.0893
|   |   |   |   |--- class: benign


The tree's very first split is on **worst perimeter** (≤ or > 105.9), followed by splits on **worst concave points**, **worst texture**, **mean radius**, and other real tumor measurements 

### Sklearn Validation

Fits scikit-learn's **DecisionTreeClassifier** with matching hyperparameters (**criterion='entropy'**, **max_depth=3**, **min_samples_split=2**) on the identical data, as an independent correctness check.

In [9]:
from sklearn.tree import DecisionTreeClassifier

sk_dt = DecisionTreeClassifier(criterion='entropy', max_depth=3, min_samples_split=2, random_state=0)
sk_dt.fit(X_train, y_train)
sk_accuracy = np.mean(sk_dt.predict(X_test) == y_test) * 100

print(f"From scratch: {accuracy:.2f}%")
print(f"sklearn:      {sk_accuracy:.2f}%")

From scratch: 91.23%
sklearn:      89.47%


**91.23%** (from scratch) vs. **89.47%** (sklearn) which is close but not identical, which is expected and fine since sklearn's tree-building includes additional internal tie-breaking rules that can lead to a slightly different but comparably good tree at the same depth.